## 1. Setup

In [26]:
from __future__ import annotations

import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple, Union



## 2. Data Structures and Parsing


In [27]:
def parse_jsp_instances(file_path):
    """
    Parses a text file containing JSP instances into a dictionary.
    Structure: { 'instance_name': { 'n_jobs': int, 'n_machines': int, 'jobs': [[(m, t), ...], ...] } }
    """
    instances = {}

    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    raw_instances = re.split(r'\s*\+{5,}\s*instance\s+', content, flags=re.IGNORECASE)

    for section in raw_instances:
        if not section.strip():
            continue

        lines = [line.strip() for line in section.split('\n') if line.strip()]
        if not lines:
            continue

        instance_name = lines[0]

        dims_index = -1
        for i, line in enumerate(lines):
            if re.match(r'^\d+\s+\d+$', line):
                dims_index = i
                break

        if dims_index == -1:
            continue

        n_jobs, n_machines = map(int, lines[dims_index].split())

        if dims_index + n_jobs >= len(lines):
            raise ValueError(f"Malformed instance {instance_name}: missing job rows")

        job_data = []
        for i in range(dims_index + 1, dims_index + 1 + n_jobs):
            values = list(map(int, lines[i].split()))
            if len(values) % 2 != 0:
                raise ValueError(f"Malformed job row in {instance_name}: odd token count")
            tasks = [(values[j], values[j + 1]) for j in range(0, len(values), 2)]
            job_data.append(tasks)

        instances[instance_name] = {
            'n_jobs': n_jobs,
            'n_machines': n_machines,
            'jobs': job_data,
        }

    return instances



## 3. Genetic Algorithm 

In [28]:
def load_problem(instances, instance_name):
    if instance_name not in instances:
        available = ', '.join(sorted(instances.keys())[:20])
        raise ValueError(f"Unknown instance '{instance_name}'. Available: {available}...")
    return instances[instance_name]


def chromosome_template(problem):
    template = []
    for job_id, ops in enumerate(problem['jobs']):
        template.extend([job_id] * len(ops))
    return template


def random_chromosome(template, rng):
    c = list(template)
    rng.shuffle(c)
    return c


def decode_chromosome(problem, chromosome):
    n_jobs = problem['n_jobs']
    n_machines = problem['n_machines']
    jobs = problem['jobs']

    machine_ready = [0] * n_machines
    job_ready = [0] * n_jobs
    next_op = [0] * n_jobs

    starts = [[0] * len(jobs[j]) for j in range(n_jobs)]
    ends = [[0] * len(jobs[j]) for j in range(n_jobs)]

    for job_id in chromosome:
        op_idx = next_op[job_id]
        if op_idx >= len(jobs[job_id]):
            continue

        machine, duration = jobs[job_id][op_idx]
        start = max(job_ready[job_id], machine_ready[machine])
        end = start + duration

        starts[job_id][op_idx] = start
        ends[job_id][op_idx] = end

        job_ready[job_id] = end
        machine_ready[machine] = end
        next_op[job_id] += 1

    for j in range(n_jobs):
        if next_op[j] != len(jobs[j]):
            raise ValueError('Chromosome does not schedule all operations')

    return {
        'makespan': max(job_ready),
        'starts': starts,
        'ends': ends,
    }


def tournament_select(population, fitness, k, rng):
    best_idx = None
    for _ in range(k):
        idx = rng.randrange(len(population))
        if best_idx is None or fitness[idx] < fitness[best_idx]:
            best_idx = idx
    return list(population[best_idx])


def multiset_order_crossover(p1, p2, required_counts, rng):
    n = len(p1)
    a, b = sorted((rng.randrange(n), rng.randrange(n)))
    child = [-1] * n

    used = {k: 0 for k in required_counts}
    for i in range(a, b + 1):
        child[i] = p1[i]
        used[p1[i]] += 1

    fill_positions = [i for i, g in enumerate(child) if g == -1]
    fill_idx = 0
    for gene in p2:
        if used[gene] < required_counts[gene]:
            child[fill_positions[fill_idx]] = gene
            used[gene] += 1
            fill_idx += 1
            if fill_idx >= len(fill_positions):
                break

    if any(g == -1 for g in child):
        missing = []
        for gene, req in required_counts.items():
            diff = req - used[gene]
            if diff > 0:
                missing.extend([gene] * diff)
        rng.shuffle(missing)
        m = 0
        for i, g in enumerate(child):
            if g == -1:
                child[i] = missing[m]
                m += 1

    return child


def swap_mutation(chromosome, mutation_rate, rng):
    if rng.random() >= mutation_rate:
        return
    i = rng.randrange(len(chromosome))
    j = rng.randrange(len(chromosome))
    chromosome[i], chromosome[j] = chromosome[j], chromosome[i]


def solve_ga(
    problem,
    population_size=200,
    generations=500,
    crossover_rate=0.9,
    mutation_rate=0.2,
    tournament_size=3,
    elite_count=2,
    seed=42,
    verbose=False,
    log_every=50,
):
    rng = random.Random(seed)
    template = chromosome_template(problem)

    required_counts = {}
    for g in template:
        required_counts[g] = required_counts.get(g, 0) + 1

    population = [random_chromosome(template, rng) for _ in range(population_size)]
    fitness = [decode_chromosome(problem, c)['makespan'] for c in population]

    best_idx = min(range(population_size), key=lambda i: fitness[i])
    best = list(population[best_idx])
    best_cost = fitness[best_idx]

    if verbose:
        print(f'Initial best makespan: {best_cost}', flush=True)

    for gen in range(1, generations + 1):
        ranked = sorted(range(population_size), key=lambda i: fitness[i])
        new_population = [list(population[i]) for i in ranked[:elite_count]]

        while len(new_population) < population_size:
            p1 = tournament_select(population, fitness, tournament_size, rng)
            p2 = tournament_select(population, fitness, tournament_size, rng)

            if rng.random() < crossover_rate:
                child = multiset_order_crossover(p1, p2, required_counts, rng)
            else:
                child = list(p1)

            swap_mutation(child, mutation_rate, rng)
            new_population.append(child)

        population = new_population
        fitness = [decode_chromosome(problem, c)['makespan'] for c in population]

        gen_best_idx = min(range(population_size), key=lambda i: fitness[i])
        if fitness[gen_best_idx] < best_cost:
            best_cost = fitness[gen_best_idx]
            best = list(population[gen_best_idx])

        if verbose and (gen % log_every == 0 or gen == generations):
            print(f'Generation {gen}/{generations}: best={best_cost}', flush=True)

    return best, best_cost



## 4. Run


In [29]:
DATA_FILE = Path('jobshop.txt')

print('Working directory:', Path.cwd())
print('jobshop.txt exists:', DATA_FILE.exists())

instances = parse_jsp_instances(DATA_FILE)
print(f'Found {len(instances)} instances')
print('First 15:', ', '.join(sorted(instances.keys())[:15]))



Working directory: /Users/nordenadler/Documents/IntOpt/ROA
jobshop.txt exists: True
Found 80 instances
First 15: abz5, abz6, abz7, abz8, abz9, ft06, ft10, ft20, la01, la02, la03, la04, la05, la06, la07


In [34]:
# Small benchmark instance with progress logs
problem_ft06 = load_problem(instances, 'ft06')

best_chrom_ft06, best_makespan_ft06 = solve_ga(
    problem_ft06,
    population_size=100,
    generations=250,
    seed=42,
    verbose=True,
    log_every=20,
)

print('Instance: ft06')
print('Jobs:', problem_ft06['n_jobs'], 'Machines:', problem_ft06['n_machines'])
print('Best makespan:', best_makespan_ft06)



Initial best makespan: 762
Generation 20/250: best=651
Generation 40/250: best=630
Generation 60/250: best=630
Generation 80/250: best=630
Generation 100/250: best=630
Generation 120/250: best=630
Generation 140/250: best=630
Generation 160/250: best=630
Generation 180/250: best=630
Generation 200/250: best=630
Generation 220/250: best=628
Generation 240/250: best=628
Generation 250/250: best=628
Instance: ft06
Jobs: 10 Machines: 5
Best makespan: 628


In [31]:
# Larger instance (this takes longer)
problem_la01 = load_problem(instances, 'la01')

best_chrom_la01, best_makespan_la01 = solve_ga(
    problem_la01,
    population_size=160,
    generations=250,
    seed=42,
    verbose=True,
    log_every=50,
)

print('Instance: la01')
print('Jobs:', problem_la01['n_jobs'], 'Machines:', problem_la01['n_machines'])
print('Best makespan:', best_makespan_la01)



Initial best makespan: 752
Generation 50/250: best=696
Generation 100/250: best=696
Generation 150/250: best=687
Generation 200/250: best=678
Generation 250/250: best=678
Instance: la01
Jobs: 10 Machines: 5
Best makespan: 678


## 6. Benchmark


In [32]:
import statistics
import time

BEST_KNOWN = {
    'ft06': 55,
    'ft10': 930,
    'ft20': 1165,
    'la01': 666,
}

INSTANCES_TO_TEST = ['ft06', 'la01']
SEEDS = list(range(20))

GA_PARAMS = {
    'population_size': 160,
    'generations': 250,
    'crossover_rate': 0.9,
    'mutation_rate': 0.2,
    'tournament_size': 3,
    'elite_count': 2,
}


def benchmark_instances(instances, instances_to_test, seeds, best_known, ga_params):
    rows = []

    for name in instances_to_test:
        problem = load_problem(instances, name)
        makespans = []
        runtimes = []

        for seed in seeds:
            t0 = time.perf_counter()
            _, best = solve_ga(problem, seed=seed, verbose=False, **ga_params)
            t1 = time.perf_counter()

            makespans.append(best)
            runtimes.append(t1 - t0)

        best_found = min(makespans)
        mean_found = statistics.mean(makespans)
        std_found = statistics.pstdev(makespans) if len(makespans) > 1 else 0.0
        avg_time_s = statistics.mean(runtimes)

        ref = best_known.get(name)
        gap_best = None if ref in (None, 0) else 100.0 * (best_found - ref) / ref

        rows.append({
            'instance': name,
            'best_known': ref,
            'best_found': best_found,
            'mean_found': round(mean_found, 2),
            'std_found': round(std_found, 2),
            'gap_best_%': None if gap_best is None else round(gap_best, 2),
            'avg_time_s': round(avg_time_s, 4),
            'runs': len(seeds),
        })

    return rows


def print_results_table(rows):
    headers = ['instance', 'best_known', 'best_found', 'mean_found', 'std_found', 'gap_best_%', 'avg_time_s', 'runs']

    data = []
    for r in rows:
        data.append([
            str(r['instance']),
            str(r['best_known']),
            str(r['best_found']),
            str(r['mean_found']),
            str(r['std_found']),
            'NA' if r['gap_best_%'] is None else str(r['gap_best_%']),
            str(r['avg_time_s']),
            str(r['runs']),
        ])

    widths = [len(h) for h in headers]
    for row in data:
        for i, v in enumerate(row):
            widths[i] = max(widths[i], len(v))

    def fmt_row(values):
        return ' | '.join(v.ljust(widths[i]) for i, v in enumerate(values))

    print(fmt_row(headers))
    print('-+-'.join('-' * w for w in widths))
    for row in data:
        print(fmt_row(row))


results = benchmark_instances(instances, INSTANCES_TO_TEST, SEEDS, BEST_KNOWN, GA_PARAMS)
print_results_table(results)



instance | best_known | best_found | mean_found | std_found | gap_best_% | avg_time_s | runs
---------+------------+------------+------------+-----------+------------+------------+-----
ft06     | 55         | 55         | 56.05      | 1.4       | 0.0        | 0.398      | 20  
la01     | 666        | 666        | 667.3      | 3.72      | 0.0        | 0.5467     | 20  
